In [27]:
import os
import random

import pandas as pd
from scipy import stats

from tqdm.notebook import tqdm

from ir_rpp.pref_eval.pref_eval import get_measures, prepare_qrels_runs, get_prefs
from ir_rpp.pref_eval.util import trec_io

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
DATASET = ("ml-1M","libraryThing","beerAdvocate")[0]
DATA_DIR = f"../../data/{DATASET}"
SCORES = ['rpp', 'dcgrpp', 'invrpp', 'ap', 'ndcg', 'rr']

qrels_file = f"{DATA_DIR}/qrels/2018.txt"
runfiles = [ f"{DATA_DIR}/runs/{fname}" for fname in os.listdir(f"{DATA_DIR}/runs") if fname.endswith(".gz")]

In [29]:
qrels, runs = prepare_qrels_runs(
    qrels_file,
    runfiles,
    binary_relevance=4,  # NOTE: threshold from article
    relevance_floor=None,
    topk=None,
    query_fraction=1.0,
    label_fraction=1.0,
    label_fraction_policy="random",
)

Reading run files:   0%|          | 0/21 [00:00<?, ?it/s]

In [30]:
print(f"requests {len(qrels)}")
print(f"rel/request {sum([ len(val)for val in qrels.values()]) / len(qrels):.2f}")

requests 6005
rel/request 18.87


In [31]:
pairwise_comparisons, raw_metrics = get_prefs(
    sample=0, runs=runs, measures=get_measures([], "all"), per_query=True
)

  0%|          | 0/6005 [00:00<?, ?it/s]

In [32]:
nb_comparisons = len(pairwise_comparisons)
count_significant = {score : 0 for score in SCORES}
PVAL_THRESHOLD = 0.05 / (nb_comparisons*len(SCORES)) # Bonferroni

for pairwise_comparison in tqdm(pairwise_comparisons.values()):
    for score in SCORES:
        res = stats.ttest_1samp(pairwise_comparison[score], popmean=0)
        if res.pvalue < PVAL_THRESHOLD:
            count_significant[score] += 1

  0%|          | 0/210 [00:00<?, ?it/s]

In [33]:
prop_significant = {score: round(count / nb_comparisons,4) for score, count in count_significant.items()}
print(pd.DataFrame([prop_significant],index=["t-test Bonferroni"]))

                      rpp  dcgrpp  invrpp      ap    ndcg      rr
t-test Bonferroni  0.9524  0.9524  0.9571  0.9333  0.9524  0.8619
